In [1]:
debts = [
    {"name": "SBI Credit Card", "bal": 50000, "apr": 42, "min_pay": 2500, "due_day": 18},
    {"name": "HDFC Home Loan",  "bal": 2500000, "apr": 8.5, "emi": 23000, "due_day": 5},
    {"name": "Axis Car",        "bal": 600000, "apr": 9.5, "emi": 13500, "due_day": 10},
    {"name": "Edu Loan",        "bal": 400000, "apr": 10.5, "emi": 0, "due_day": None},  # moratorium
]

extra_payments = {
    "2025-11-10": {"SBI Credit Card": 5000},   # Diwali bonus
    "2025-12-01": {"HDFC Home Loan": 20000},
}

In [2]:
from datetime import datetime, timedelta
import pandas as pd

start = datetime(2025, 11, 4)
days = 180
rows = []

for day_offset in range(days):
    today = start + timedelta(days=day_offset)
    dd = today.day
    mm = today.strftime("%b-%y")

    for d in debts:
        bal = d["bal"]
        if bal <= 0: continue

        # 1. Daily interest
        daily_rate = d["apr"] / 36500
        interest = bal * daily_rate
        d["bal"] += interest                 # accrue today
        rows.append([mm, today.strftime("%d"), d["name"], "Interest", interest])

        # 2. Extra payment today?
        key = today.strftime("%Y-%m-%d")
        if key in extra_payments and d["name"] in extra_payments[key]:
            amt = extra_payments[key][d["name"]]
            d["bal"] -= amt
            rows.append([mm, today.strftime("%d"), d["name"], f"Extra ₹{amt}", -amt])

        # 3. Scheduled payment
        if d["due_day"] and dd == d["due_day"]:
            if "min_pay" in d:
                pay = max(d["min_pay"], bal * 0.05)  # credit-card 5% rule
            else:
                pay = d["emi"]
            pay = min(pay, d["bal"])                 # don’t overpay
            d["bal"] -= pay
            rows.append([mm, today.strftime("%d"), d["name"], "EMI/MinPay", -pay])

In [3]:
df = pd.DataFrame(rows, columns=["Month", "Day", "Debt", "Event", "₹"])
df["₹"] = df["₹"].round(2)

print("YOUR 6-MONTH DEBT DIARY")
print(df.to_string(index=False))

print("\nFINAL BALANCES")
for d in debts:
    print(f"{d['name']:20} → ₹{d['bal']:,.0f}")

total_interest = df[df["Event"]=="Interest"]["₹"].sum()
print(f"\nTotal interest you WILL pay → ₹{total_interest:,.0f}")

YOUR 6-MONTH DEBT DIARY
 Month Day            Debt        Event         ₹
Nov-25  04 SBI Credit Card     Interest     57.53
Nov-25  04  HDFC Home Loan     Interest    582.19
Nov-25  04        Axis Car     Interest    156.16
Nov-25  04        Edu Loan     Interest    115.07
Nov-25  05 SBI Credit Card     Interest     57.60
Nov-25  05  HDFC Home Loan     Interest    582.33
Nov-25  05  HDFC Home Loan   EMI/MinPay -23000.00
Nov-25  05        Axis Car     Interest    156.21
Nov-25  05        Edu Loan     Interest    115.10
Nov-25  06 SBI Credit Card     Interest     57.67
Nov-25  06  HDFC Home Loan     Interest    577.11
Nov-25  06        Axis Car     Interest    156.25
Nov-25  06        Edu Loan     Interest    115.13
Nov-25  07 SBI Credit Card     Interest     57.73
Nov-25  07  HDFC Home Loan     Interest    577.24
Nov-25  07        Axis Car     Interest    156.29
Nov-25  07        Edu Loan     Interest    115.17
Nov-25  08 SBI Credit Card     Interest     57.80
Nov-25  08  HDFC Home Loan